## Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torch.optim import Adam

## Creating the training dataset

In [5]:
# sample corpus for training the Transformer Encoder-Decoder model
token_to_id = {'what': 0, 
               'is': 1,
               'your': 2,
               'name': 3,
               'Samuel': 4,
               '<EOS>': 5,
               }

id_to_token = {v: k for k, v in token_to_id.items()}
print(token_to_id)
print(id_to_token)

{'what': 0, 'is': 1, 'your': 2, 'name': 3, 'Samuel': 4, '<EOS>': 5}
{0: 'what', 1: 'is', 2: 'your', 3: 'name', 4: 'Samuel', 5: '<EOS>'}


In [8]:
inputs = torch.tensor([[token_to_id['what'],
                       token_to_id['is'],
                       token_to_id['your'],
                       token_to_id['name'],
                       token_to_id['<EOS>'],
                       token_to_id['Samuel']],

                       [token_to_id['name'],
                        token_to_id['is'],
                        token_to_id['what'],
                        token_to_id['your'],
                        token_to_id['<EOS>'],
                        token_to_id['Samuel']]])

labels = torch.tensor([[token_to_id['what'],
                        token_to_id['is'],
                        token_to_id['your'],
                        token_to_id['name'],
                        token_to_id['<EOS>'],
                        token_to_id['Samuel']],

                       [token_to_id['name'],
                        token_to_id['is'],
                        token_to_id['what'],
                        token_to_id['your'],
                        token_to_id['<EOS>'],
                        token_to_id['Samuel']]])

In [14]:
dataset = TensorDataset(inputs, labels)
dataloader = DataLoader(dataset)

for batch in dataloader:
    print(batch)

[tensor([[0, 1, 2, 3, 5, 4]]), tensor([[0, 1, 2, 3, 5, 4]])]
[tensor([[3, 1, 0, 2, 5, 4]]), tensor([[3, 1, 0, 2, 5, 4]])]


## Position Encoding

In [16]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=6):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_len, d_model) # positional encoding matrix
        position = torch.arange(start=0, end=max_len, step=1, dtype=torch.float).unsqueeze(1) # position indices for each row
        embedding_index = torch.arange(start=0, end=d_model, step=2, dtype=torch.float) # embedding dimension indices for each column

        div_term = 1 / torch.tensor(10000.0) ** (embedding_index / d_model) # scaling factor for each embedding dimension

        pe[:, 0::2] = torch.sin(position * div_term) # apply sine to even columns
        pe[:, 1::2] = torch.cos(position * div_term) # apply cosine to odd columns

        self.register_buffer('pe', pe) # register as buffer to avoid updating during training

    def forward(self, word_embeddings):
        return word_embeddings + self.pe[:word_embeddings.size(1), :] # add positional encoding to input sequence

In [28]:
positional_encoding = PositionalEncoding(d_model=2, max_len=6)
word_embeddings = torch.randn(2, 6, 2) # batch size 2, sequence length 6, embedding dimension 2
encoded_embeddings = positional_encoding(word_embeddings)
print(f"Shape of encoded_embeddings[0]:\n{encoded_embeddings[0].shape}")
print(f"\nFirst encoded embedding:\n{encoded_embeddings[0]}")

Shape of encoded_embeddings[0]:
torch.Size([6, 2])

First encoded embedding:
tensor([[ 1.0144,  0.6231],
        [ 1.0343,  1.5211],
        [ 4.0519,  0.0403],
        [ 2.2661,  1.1211],
        [ 0.9498, -0.6091],
        [-0.5234,  0.8063]])


## Attention

In [29]:
class Attention(nn.Module):
    def __init__(self, d_model=2):
        super(Attention, self).__init__()

        self.d_model = d_model
        self.W_q = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_k = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_v = nn.Linear(in_features=d_model, out_features=d_model, bias=False)

        self.row_dim = 0
        self.col_dim = 1

    def forward(self, query, key, value, mask=None):
        Q = self.W_q(query) # (batch_size, seq_len, d_model)
        K = self.W_k(key) # (batch_size, seq_len, d_model)
        V = self.W_v(value) # (batch_size, seq_len, d_model)

        sims = torch.matmul(Q, K.transpose(dim0=self.row_dim, dim1=self.col_dim)) # (batch_size, seq_len, seq_len)

        scaled_sims = sims / torch.sqrt(torch.tensor(K.size(self.col_dim), dtype=torch.float))

        if mask is not None:
            scaled_sims = scaled_sims.masked_fill(mask=mask, value=1e-9)

        attention_percents = F.softmax(scaled_sims, dim=self.col_dim)
        attention_scores = torch.matmul(attention_percents, V)
        
        return attention_scores

In [37]:
word_embeddings[0].shape

torch.Size([6, 2])

In [40]:
query = torch.rand(1, 6, 2) # (batch_size, seq_len, d_model)
key = torch.rand(1, 6, 2)   # (batch_size, seq_len, d_model)
value = torch.rand(1, 6, 2)  # (batch_size, seq_len, d_model)
print(query.shape)
print(query[0].shape)

torch.Size([1, 6, 2])
torch.Size([6, 2])


In [ ]:
attention = Attention(d_model=2)

output = attention(query[0], key[0], value[0])
print(output)

tensor([[0.0413, 0.1277],
        [0.0380, 0.1278],
        [0.0404, 0.1277],
        [0.0384, 0.1278],
        [0.0401, 0.1277],
        [0.0399, 0.1277]], grad_fn=<MmBackward0>)
